# inplace-op-unsafe-warning — worked example 1: mul_inplace_safe refuses non-leaf tensors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-op-unsafe-warning`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

An in-place op overwrites a tensor's underlying `.array`. If that tensor is a non-leaf (its `.recipe` is not `None`), its array is also cached inside downstream nodes' Recipes as a parent value. Mutating it silently corrupts the values the reverse pass will read. The safe rule: refuse the mutation whenever `x.recipe is not None`, and only allow it for leaves.

## Worked solution

We want an in-place multiply `x *= y` that protects the compute graph.

1. **Check the guard first.** A tensor is a graph intermediate exactly when it carries a Recipe (`x.recipe is not None`). Those are the dangerous ones, because some other node has stashed `x.array` as an argument it will need during backprop. So we test `x.recipe is not None` and `raise RuntimeError` with the word 'in-place' in the message.
2. **Leaf path.** If `x.recipe is None`, the tensor is a leaf (an input or a parameter). No downstream Recipe points at it, so overwriting its storage is harmless. We do `x.array *= y.array` and return the same object.
3. **Return identity.** In-place semantics mean the caller's handle must still point at the mutated tensor, so we return `x` itself, not a new wrapper.

The demo builds a leaf and mutates it successfully, then builds a non-leaf (by attaching a Recipe) and shows the guard fires.

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def mul_inplace_safe(x: MiniTensor, y: MiniTensor) -> MiniTensor:
    # Guard: a tensor carrying a Recipe is a graph intermediate; its
    # array is cached elsewhere, so an in-place op would corrupt backprop.
    if x.recipe is not None:
        raise RuntimeError(
            'in-place op forbidden on a Tensor with a recipe — '
            'would corrupt cached values on the graph'
        )
    x.array *= y.array
    return x

# leaf path: succeeds, same object back
leaf = MiniTensor([2.0, 3.0, 4.0], requires_grad=True)
y = MiniTensor([10.0, 10.0, 10.0])
out = mul_inplace_safe(leaf, y)
print('leaf mutated in place:', out.array, 'same object:', out is leaf)

# non-leaf path: guard fires
node = MiniTensor([1.0, 1.0], recipe=Recipe(np.add, (), {}, {}))
try:
    mul_inplace_safe(node, MiniTensor([5.0, 5.0]))
except RuntimeError as e:
    print('guard fired:', 'in-place' in str(e))